# Introducción

Este notebook funciona ***exclusivamente*** para consultas empleando *nombres*. El notebook que funciona con SMILES está en `busquedaSmiles.ipynb`

In [ ]:
'''Primeramente, vamos a instalar las dependencias necesarias para realizar
las consultas dentro de la base de datos LOTUS'''
%pip install pandas
%pip install rdkit py3Dmol

In [ ]:
'''Luego, se importan las bibliotecas Requests, Pandas, RDKit y sus módulos Chem y Draw'''
import pandas as pd
import requests
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.drawOptions.addAtomIndices = False
IPythonConsole.drawOptions.addStereoAnnotation = True


Podemos hacer una búsqueda sencilla empleando el nombre de un producto natural como por ejemplo, el mentol, presente en productos cosméticos como labiales, esencias y el famoso Vaporrub.

> Los nombres de los compuestos deben estar exclusivamente en inglés

Aquí doy unos cuantos que se pueden usar:
- Quercetin
- Cyanidin
- Solanapyrone
- Menthol
- Carvone
- Caffeine
- Delphinidin
- Limonene
- Celastrol
- Catechol
- Cinnamaldehyde

In [ ]:
'''Búsqueda por nombre o especie:

Nos van a solicitar el nombre de un producto natural, para posteriormente ejecutar un query simple a la API REST de LOTUS.
Este bloque nos va a devolver el status del query y la estrucutra química del compuesto que estamos buscando, además de 
estructuras que son similares'''

# En esta primera parte, quise cachar una excepción de End of File
while True:
    try:
        compound=input("Introduce el nombre de un producto natural (en inglés):")
        if not compound:
            raise EOFError
    except EOFError:
        print("No se introdujo un compuesto, inténtalo de nuevo ;)")
        break
    else:
        # Acá, quiero cachar un error en la conexión mediante la API REST
        try:
            url=f"https://lotus.naturalproducts.net/api/search/simple?query={compound}"
            response=requests.get(url,timeout=1500)
            data=response.json()
            if response.status_code != 200:
                raise Exception
        except Exception:
            print("Error en el query. Tu petición es inválida o tardó demasiado.")
            break
        print(f"Status code del query: {response.status_code}")
        df=pd.json_normalize(data,record_path=['naturalProducts'],meta=['originalQuery','determinedInputType'])
        smiles=df.at[0,'smiles']
        mol=Chem.MolFromSmiles(smiles)
        img=Draw.MolToImage(mol,size=(300,300),legend=df['traditional_name'].iloc[0])
        print(f"Estructura química del query:")
        display(img)
        break

In [ ]:
''' Este es el armado de nuestro query en 3D, esta celda acepta tanto el output de un query por nombre como
del query por código SMILES'''

from rdkit.Chem import AllChem, rdCoordGen
from rdkit.Chem.Draw import IPythonConsole 
IPythonConsole.ipython_useSVG=True
import py3Dmol

'''Armado, inserción de átomos de hidrógeno y optimización con MMFF94
La función recibe un objeto mol de RDKit, para posteriormente generar
coordenadas XYZ para cada átomo que constituye la molécula'''

def mol3d(mol):
    mol=Chem.MolFromSmiles(smiles)
    rdCoordGen.AddCoords(mol)
    mol=Chem.AddHs(mol,addCoords=True)
    AllChem.EmbedMolecule(mol)
    AllChem.MMFFOptimizeMolecule(mol)
    return mol
mol=mol3d(mol=smiles)

view = py3Dmol.view(
    data=Chem.MolToMolBlock(mol),  # Convert the RDKit molecule for py3Dmol
    style={"stick": {}, "sphere": {"scale": 0.3}}
)
view.zoomTo()

In [ ]:
'''
Organizamos los datos en un dataframe de Pandas

LOTUS como base de datos tiene un tamaño de 276,521 productos naturales registrados,
creciendo conforme los científicos van reportando estructuras novedosas.

Nota: En química, es usual encontrar compuestos con el mismo peso molecular pero 
con estructura diferente. Inicialmente, añadí un drop_duplicates para eliminar entradas
con el mismo peso molecular, pero vi que habían demasiados isómeros en algunos queries.

'''
df=pd.json_normalize(data['naturalProducts'])
print("¿Cuántas filas y columnas tiene el archivo antes de la limpieza?","\n",df.shape[0],"filas y", df.shape[1],"columnas\n")

# Limpieza de los datos
df=df.dropna(axis=1,how='all')
df.drop(list(df.filter(regex="fragment*")),axis=1,inplace=True)
#df=df.filter(['id','lotus_id','traditional_name','iupac_name','molecular_weight','total_atom_number','contains_sugar','smiles',])

# Resumen final
print("¿Cuántas filas y columnas tiene el archivo tras de la limpieza?","\n",df.shape[0],"filas y", df.shape[1],"columnas\n")
print("Esta es la tabla tras la limpieza:")
display(df.sort_index(axis=0))

In [ ]:
'''
LOTUS también tiene datos referentes a la clasificación biológica de los compuestos, pero cuando 
se añadían al DataFrame original, aparecían como 'taxonomyReferenceObjects$x$x$'....' anidados dentro del
data frame principal y encontré que tenía que iterar adentro de ese diccionario y convertirlo con
numpy porque pandas no tiene una herramienta para tablas anidadas como json_normalize, y la API REST
no tiene una opción que solo responda con la taxonomía.

En esta celda, lo que se hace es crear una tabla alterna extrayendo la taxonomía del compuesto a partir del
data frame original para encontrar las especies y la familia a la que pertenecen según el 'lotus_id' 
de los resultados de nuestro query, para luego hacer un join y así ir quitando duplicados y generar 
una nueva tabla resultado.
'''

# Creamos un nuevo Data Frame de taxonomía a partir del df original
rows = []
for producto in data['naturalProducts']:
    for ref, dbs in producto['taxonomyReferenceObjects'].items():
        for db, organisms in dbs.items():
            for org in organisms:
                rows.append({'lotus_id': producto['lotus_id'],
                             'reference': ref.replace('$x$x$', '.'),
                             'taxonomy_db': db,
                             **org})

taxonomy = pd.DataFrame(rows)

# Una vez listo, usamos taxonomy para crear result, que es una
# version corregida del dataframe original sin los defectos de
# la columna taxonomyReferenceObjects original. Para ello, usamos
# un right join, porque algunos registros carecen de informacion 
# taxonomica pero aun cuentan con un id

result=pd.merge(df,taxonomy,on='lotus_id',how='right',sort=True)
result=result.drop_duplicates('species')
result=result.dropna(how='all') # Si pongo any, me borra todo

# De todos modos, checamos que la tabla no esté vacía y aprovechamos
# obtener los conteos del compuesto según la especie y la familia 
# biologica, sea planta, animal u hongo

if not result.empty:
    fam=result['family'].value_counts()
    gen=result['genus'].value_counts()

print(f'Nuestro query \"{compound}\" se encuentra en la siguiente tabla de especies y familias vegetales')
print(f"Siendo más frecuente en las familia {fam.index[0]} ({fam.iloc[0]}) coincidencias)")
print(f"Y en el género {gen.index[0]} ({gen.iloc[0]} coincidencias)\n")
print("¡Mega dato botánico!:\n\tQue un compuesto sea más frecuente dentro de una familia\n\tno significa que lo sea en alguno de sus géneros.")
cols=['traditional_name','lotus_id','family','genus','species','taxonomy_db']
display(result.filter(items=cols).sort_values(by=['family','genus','species']))


In [ ]:
# Obtener los productos naturales de la respuesta JSON
productos = data.get('naturalProducts', [])

# Construyes los registros de una única tabla
rows = []

for producto in productos:

    # Separamos la información general de la información taxonómica
    producto_base = producto.copy()
    taxonomia = producto_base.pop('taxonomyReferenceObjects', {})

    # Aplanamos los datos generales del producto
    datos_producto = pd.json_normalize(producto_base, sep='_').iloc[0].to_dict()

    # Recorremos los diccionarios de taxonomía
    for referencia, bases_datos in taxonomia.items():
		# Usamos 2 for para que recorra los elementos anidados de tu API
        for base_datos, organismos in bases_datos.items():

            for organismo in organismos:
                # Aplanamos los datos del organismo
                datos_organismo = pd.json_normalize(
                    organismo, sep='_'
                ).iloc[0].to_dict()

                # Unimos los datos en un mismo registro
                registro = {
                    **datos_producto,
                    **datos_organismo,
                    'reference': referencia.replace('$x$x$', '.'),
                    'taxonomy_db': base_datos
                }
                rows.append(registro)
    # Si el producto no tiene información taxonómica,
    # conservamos también su información general
    if not taxonomia:
        rows.append(datos_producto)

# Creamos una sola tabla
df2 = pd.DataFrame(rows)

# Eliminamos columnas completamente vacías
df2 = df.dropna(axis=1, how='any')

print("Dimensiones de nuestra tabla:", df.shape)
display(df2.head())

Algunos productos naturales existen como **agliconas** y **gliconas**, siendo una aglicona la forma sin azúcar y la glicona la forma unida a uno o más azúcares. 

Ejemplos de productos naturales de esta naturaleza son:

- **Antocianinas y antocianidinas**: Las antocinaninas son compuestos responsables de colores rojizos, morados, rosas y azulados en flores, frutos y hojas de algunas especies de plantas. Estos compuestos son sensibles a la acidez del suelo y, en flores como las dahlias y las petuñas, el color de la flor es indicativo del pH aproximado del suelo. Por otro lado, las antocianinas son la forma de glicona de las antocianidinas, estas últimas siendo la forma sin azúcares de las antocianinas. Ejemplos son la delfinidina (aglicona) y la delfinidina 3-glucósido (glicona).

- **Glicósidos cardiotónicos**: Los glicósidos cardiotónicos son compuestos derivados de una aglicona de esterol vegetal unida a azúcares, tienen un intensivo uso en la medicina como tratamiento para afecciones cardíacas. Ejemplos de glicósidos cardiotónicos son la digoxina (*digoxin*) y la digitoxina (*digitoxin*), de la planta *Digitalis purpurea* o digital.

En la siguiente celda, haremos un agrupamiento según la si hay o no azúcares unidos a las estructuras dentro de nuestro data frame. Esta celda arrojará 0 si la columna `'contains_sugar'` está vacía.

> Nota:
> LOTUS no asigna siempre correctamente si un compuesto contiene azúcares. En algunos queries sí lo detecta.

In [ ]:
#display(df.groupby('contains_sugar').count()['lotus_id'])
print(f"En la tabla, tenemos {df[df['contains_sugar'] != 0]['id'].count()} producto(s) natural(es) en su forma de glicona\ny {df[df['contains_sugar'] == 0]['id'].count()} producto(s) natural(es) en su forma algicona ")

In [ ]:
'''
La ventaja de una base de datos como LOTUS es que permite encontrar compuestos con estructuras similares
al de nuestro interés.
Esta celda lo que hace es rescatar todos los códigos SMILES dentro de nuestro data frame y los acomoda en una
imagen compuesta por todos los resultados similares al query (maxhits=int)
'''
mol =  [ Chem.MolFromSmiles(x) for x in df['smiles']]
names=list(df['traditional_name'])
img=Draw.MolsToGridImage(mol,molsPerRow=5,legends=names,subImgSize=(300,300))  
print(f"Estructuras moleculares similares a nuestro query \"{smiles}\"")
print(df['traditional_name'])
img